In [ ]:
%pip install pulp

In [4]:
import pulp

# 1. DATA DEFINITIE
# De rijen (Stock / X) en kolommen (Constructie / Y)
stock_items = ['X1', 'X2', 'X3', 'X4N', 'X5N']
construction_slots = ['Y1', 'Y2', 'Y3', 'Y4', 'Y5', 'Y6']

# Jouw Cost Matrix (zoals je die in de tabel gaf)
# Let op: de volgorde moet overeenkomen met stock_items
cost_matrix = [
    [2,  6,  8,  3,  1,  4],   # X1
    [4,  5,  3,  6,  3,  2],   # X2
    [1,  7,  4,  9,  6,  5],   # X3
    [10, 10, 2,  2,  10, 10],  # X4 (Nieuw)
    [10, 1,  1,  10, 10, 10]   # X5 (Nieuw)
]

# We maken een dictionary om makkelijk kosten op te zoeken: costs['X1']['Y1'] = 2
costs = {
    stock_items[i]: {construction_slots[j]: cost_matrix[i][j]
                     for j in range(len(construction_slots))}
    for i in range(len(stock_items))
}

# 2. HET MODEL OPZETTEN
# We willen de kosten MINIMALISEREN
prob = pulp.LpProblem("Reclaimed_Timber_Matching", pulp.LpMinimize)

# 3. DECISION VARIABLES
# Dit maakt voor elke combinatie X-Y een variabele die 0 of 1 kan zijn (Binary)
# x['X1']['Y1'] wordt 1 als we matchen, anders 0.
x = pulp.LpVariable.dicts("Match", (stock_items, construction_slots), 0, 1, pulp.LpBinary)

# 4. OBJECTIVE FUNCTION (Doel)
# Som van alle (kosten * keuze)
prob += pulp.lpSum([x[i][j] * costs[i][j] for i in stock_items for j in construction_slots])

# 5. CONSTRAINTS (De Regels)
# Regel A: Elke Y (constructie onderdeel) MOET precies 1 stuk hout krijgen.
for j in construction_slots:
    prob += pulp.lpSum([x[i][j] for i in stock_items]) == 1

# Regel B: Capaciteit van de Stock (X)
# X1, X2, X3 zijn 'reclaimed' en uniek -> Maximaal 1x gebruiken
reclaimed_limit = 1
for i in ['X1', 'X2', 'X3']:
    prob += pulp.lpSum([x[i][j] for j in construction_slots]) <= reclaimed_limit

# X4, X5 zijn 'nieuw' -> Mogen vaker gebruikt worden (bijv. max 6x, want er zijn 6 gaten)
new_limit = 10000
for i in ['X4N', 'X5N']:
    prob += pulp.lpSum([x[i][j] for j in construction_slots]) <= new_limit

# 6. OPLOSSEN
prob.solve()

# 7. RESULTAAT PRINTEN
print(f"Status: {pulp.LpStatus[prob.status]}")
print("-" * 35)

total_cost = 0
print(f"{'Stock':<10} -> {'Element':<10} {'Kosten'}")

for j in construction_slots:
    for i in stock_items:
        if x[i][j].varValue == 1: # Als de beslissing 'JA' is
            print(f"{j:<10} -> {i:<10} {costs[i][j]}")
            total_cost += costs[i][j]

print("-" * 35)
print(f"Totale 'Penalty' Cost: {total_cost}")

Status: Optimal
-----------------------------------
Stock      -> Element    Kosten
Y1         -> X3         1
Y2         -> X5N        1
Y3         -> X5N        1
Y4         -> X4N        2
Y5         -> X1         1
Y6         -> X2         2
-----------------------------------
Totale 'Penalty' Cost: 8
